In [1]:
!pip install -U langchain langchain-core langchain-community langchain-google-genai streamlit pyngrok bs4 pypdf chromadb faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 6.3 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [2]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,165 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Fetched 13.8 MB in 3s (4,328 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
R

In [3]:
import subprocess
import time

print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

Starting Ollama server...


In [4]:
print("Downloading llama2 model (this may take a few minutes)...")
!ollama pull llama2
print("Done! Ollama is now ready to serve queries.")


Done! Ollama is now ready to serve queries.


##### Data Ingestion

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("rag_test.txt")
text_documents = loader.load()
text_documents

/tmp/ipykernel_34253/2712929328.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'rag_test.txt'}, page_content='================================================================================\nDOCUMENT TITLE: Aetheris Dynamics - Internal Operations & Product Manual (2026)\nDOCUMENT ID: AD-OPS-2026-V4\nCLASSIFICATION: Internal / Engineering & Support\n================================================================================\n\n1. COMPANY OVERVIEW & HISTORY\nAetheris Dynamics was founded on March 14, 2019, in Zurich, Switzerland, by Dr. Elena Rostova and Marcus Vance. Originally starting as a boutique research lab focusing on solid-state battery electrolytes, the company pivoted in 2022 to autonomous atmospheric drone networks. \n\nAs of Q3 2026, Aetheris Dynamics employs 412 full-time staff across three primary hubs:\n- Zurich HQ (R&D and Executive Leadership): 185 employees\n- Austin, Texas (Manufacturing & Flight Testing): 160 employees\n- Singapore (APAC Logistics & Enterprise Sales): 67 employees\n\nThe company\'s current Ch

In [7]:
import os

os.environ['GEMINI_API_KEY'] = "<api-key>"

##### Web based loader

In [8]:
from langchain_community.document_loaders import WebBaseLoader

import bs4

## load, chunk and index the content of the html page

loader = WebBaseLoader(
    web_path = ("https://lilianweng.github.io/posts/2023-06-23-agent/"),
    bs_kwargs = dict(parse_only = bs4.SoupStrainer(
        class_ = ("post-title", "post-content", "post-header")
    )),
)

text_documents2 = loader.load()
text_documents2

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistake

##### PYPDF Loader

In [21]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('/content/attention.pdf')

docs = loader.load()
docs

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

##### Dividing into Chunks

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_spliters = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

documents = text_spliters.split_documents(docs)

documents[:3]

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

##### Vector Embedding and Vector store

```
Chromadb
```



In [23]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize local HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Build the Chroma vector store
db = Chroma.from_documents(documents, embeddings)
print("Chroma Vector Store created successfully using local HuggingFace Embeddings!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chroma Vector Store created successfully using local HuggingFace Embeddings!


In [27]:
query = "An attention function can be described as mapping a query ?"

result = db.similarity_search(query)

result[0].page_content

'around each of the sub-layers, followed by layer normalization. We also modify the self-attention\nsub-layer in the decoder stack to prevent positions from attending to subsequent positions. This\nmasking, combined with fact that the output embeddings are offset by one position, ensures that the\npredictions for position i can depend only on the known outputs at positions less than i.\n3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\nof the values, where the weight assigned to each value is computed by a compatibility function of the\nquery with the corresponding key.\n3.2.1 Scaled Dot-Product Attention\nWe call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of\nqueries and keys of dimension dk, and values of dimension dv . We compute the dot products of the\n3'


```
FAISS DB
```





In [30]:
from langchain_community.vectorstores import FAISS

db2 = FAISS.from_documents(documents, embeddings)

In [33]:
query2 = "An attention function can be described as mapping a query ?"

result2 = db2.similarity_search(query2)

result2[0].page_content

'around each of the sub-layers, followed by layer normalization. We also modify the self-attention\nsub-layer in the decoder stack to prevent positions from attending to subsequent positions. This\nmasking, combined with fact that the output embeddings are offset by one position, ensures that the\npredictions for position i can depend only on the known outputs at positions less than i.\n3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\nof the values, where the weight assigned to each value is computed by a compatibility function of the\nquery with the corresponding key.\n3.2.1 Scaled Dot-Product Attention\nWe call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of\nqueries and keys of dimension dk, and values of dimension dv . We compute the dot products of the\n3'